# 10 · End-to-end reproducibility pipeline

Master compact workflow: validate CSV, derive seed-level statistics, inspect routing/entropy, regenerate publication-style telemetry figures, summarize discrepancies, and write a Markdown report. It does **not** retrain the neural benchmarks.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
# 1) Audit
assert len(df)==1500 and df.groupby("seed").size().eq(300).all()
assert df.groupby("seed").step.apply(lambda x:set(x)==set(range(1,301))).all()
assert not df.isna().any().any() and np.allclose(df.fast_usage+df.slow_usage,1,atol=1e-5)
by=df.groupby("seed").agg(avg_loss=("loss","mean"),final_loss=("loss","last"),fast=("fast_usage","mean"),slow=("slow_usage","mean"),entropy=("entropy","mean"))
display(by)
print("VALIDATION PASS | seeds",sorted(df.seed.unique()),"| rows",len(df))

In [ ]:
# 2) Summary reconciliation and plot
fig,axs=plt.subplots(1,3,figsize=(13,3.7))
for col,title,color in [("loss","Loss","#355c8a"),("fast_usage","Fast usage","#c47a32"),("entropy","Routing entropy","#50845d")]:
    a=df.groupby("step")[col].agg(["mean","std"]); ci=2.7764451051977987*a["std"]/np.sqrt(5)
    axis=axs[("loss","fast_usage","entropy").index(col)]
    axis.plot(a.index,a["mean"],color=color)
    axis.fill_between(a.index,a["mean"]-ci,a["mean"]+ci,color=color,alpha=.2)
    axis.set(title=title,xlabel="step")
plt.tight_layout(); plot_path=Path.cwd()/"diina_pipeline_telemetry.png"; fig.savefig(plot_path,dpi=180); print("Plot saved:",plot_path.resolve())
print("JSON summary avg_loss:",summary["avg_loss"],"; telemetry pooled mean:",df.loss.mean())

In [ ]:
# 3) Findings report from actual data and source text
seedj=json.loads((PROJECT/"result_seed_2022.json").read_text()); source=(PROJECT/"DIINA_v19_1_LongDistance.py").read_text()
report=f"""# DIINA reproducibility run

- CSV rows: {len(df)}; seeds: {sorted(df.seed.unique())}; steps per seed: 300.
- Empirical pooled mean loss: {df.loss.mean():.8f}; JSON avg_loss mean: {summary['avg_loss']['mean']:.8f}.
- Mean Fast/Slow usage: {df.fast_usage.mean():.4f} / {df.slow_usage.mean():.4f}; mean entropy: {df.entropy.mean():.4f}.
- Seed 2022 telemetry average/final: {df[df.seed==2022].loss.mean():.8f} / {df[df.seed==2022].sort_values('step').loss.iloc[-1]:.8f}.
- Seed JSON average/final: {seedj['avg_loss']:.8f} / {seedj['final_loss']:.8f}.
- Benchmark v19.1 includes GRU: {'nn.GRU' in source}. Mamba is a GRU proxy; DIINA is single-GRU + sigmoid gate, unlike manuscript dual-stream LIG design.
- This pipeline analyzes logs and draws plots; it does not claim to reproduce training accuracy or benchmark model results.
"""
report_path=Path.cwd()/"diina_reproducibility_report.md"; report_path.write_text(report,encoding="utf-8"); print(report); print("Report saved:",report_path.resolve())
assert plot_path.exists() and report_path.exists()

The report and PNG are generated in the notebook working directory. To reproduce neural experiment accuracies, run the supplied Python benchmark scripts in a compatible PyTorch environment; those training runs are intentionally not part of this lightweight pipeline.